# Zepto Analytics Pipeline

This notebook covers the analytics pipeline, which profiles the customer/order dataset end-to-end and builds a predictive model to estimate delivery times.

## 1. Data Loading and Profiling

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Load data from the SQLite relational store
conn = sqlite3.connect('../zepto.db')
orders_df = pd.read_sql('SELECT * FROM orders', conn)
conn.close()

print("Dataset Shape:", orders_df.shape)
orders_df.head()

In [ ]:
# Profiling: Summary Statistics
orders_df.describe()

### Exploratory Data Analysis (EDA)

In [ ]:
# Distribution of Delivery Times
plt.figure(figsize=(8, 5))
sns.histplot(orders_df['delivery_time_mins'], bins=30, kde=True)
plt.title('Distribution of Delivery Times (mins)')
plt.xlabel('Delivery Time (mins)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Impact of Weather on Delivery Time
plt.figure(figsize=(8, 5))
sns.boxplot(x='weather', y='delivery_time_mins', data=orders_df)
plt.title('Delivery Time by Weather Condition')
plt.show()

## 2. Predictive Modeling
We will build a model to predict the `delivery_time_mins` based on the `distance_km` and `weather` conditions.

In [ ]:
# Data Preparation
# Convert categorical 'weather' into dummy variables
model_df = pd.get_dummies(orders_df, columns=['weather'], drop_first=True)

# Define features (X) and target (y)
X = model_df[['distance_km', 'weather_Rain', 'weather_Traffic']]
y = model_df['delivery_time_mins']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training Set:", X_train.shape)
print("Testing Set:", X_test.shape)

In [ ]:
# Train a Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predictions
y_pred = rf_model.predict(X_test)

## 3. Evaluation and Interpretation

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.2f} mins")
print(f"R-squared (R2): {r2:.2f}")

### Conclusion
- The Random Forest model provides a reliable estimate of delivery times based on distance and weather.
- As shown in the EDA, bad weather and traffic significantly impact the delivery SLA.
- This model can be deployed as an API service to predict delivery times for customers on checkout.